In [1]:
#!pip install langchain
#!pip install langchain_community
#!pip install langchain_text_splitter
#!pip install langchain_chroma
#!pip install langchain_groq

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [43]:
# GROQ API KEY setting
import os
os.environ["GROQ_API_KEY"] = "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"


In [44]:
# Load the PDF from internet URL
import requests
url="http://gvpcew.ac.in/Material/IT/3%20IT%20Adv%20Java.pdf"
response=requests.get(url)

In [45]:
# Save the PDF in Local file
with open("java_collections.pdf", "wb") as f:
    f.write(response.content)

In [46]:
# Load the Unstructure PDF
#!pip install unstructured
#!pip install "unstructured[pdf]"
from langchain_community.document_loaders import UnstructuredFileLoader
loader = UnstructuredFileLoader("java_collections.pdf")

In [47]:
#Split the document / chunks and load and split
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)
document = loader.load()
texts = text_splitter.split_documents(document)


In [48]:
type(texts)
len(texts)
texts[4]

Document(metadata={'source': 'java_collections.pdf'}, page_content='a2.add(2);\n\na2.add(7);\n\na1.removeAll(a2); //remove all elements from a1 if they exists in a2 SOP(a1); // O/P: 5 a1.clear(); // remove all elements from a1 and O/P: [ ]\n\n2.List(I): List is the child interface of Collection interface. If we want to represent a group of individual objects as single entity where duplicates are allowed and insertion order must be preserved, then we go for List.\n\n1. ArrayList(C): ArrayList is a class presents in java.util package which extends(inherits) from AbstractList class and implements from List, RandomAccess, Cloneable and Serializable interfaces. i) Properties: \uf09f The underlying DS is resizable array or growable array. \uf09f Duplicates are allowed and Insertion order is preserved. \uf09f Heterogeneous elements are allowed and null insertion is possible. \uf09f ArrayList is the best choice if our frequent operation is retrieval operation because ArrayList implements Rando

In [49]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings()


/tmp/ipykernel_6379/2536236937.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [50]:
from langchain_chroma import Chroma
persist_directory = "vector_db"

vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)


In [53]:
# retriever vector to text
retriever = vectordb.as_retriever()

In [54]:
# llm from groq
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [55]:
from langchain_classic.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [59]:
# invoke the qa chain and get a response for user query
query = "get me the key of collection framework and count total how many key from this pdf "
response = qa_chain.invoke({"query": query})

In [60]:
print(response)

{'query': 'get me the key of collection framework and count total how many key from this pdf ', 'result': 'According to the provided text, the 9 Key Interfaces of Collection Framework are:\n\n1. Collection (1.2)\n2. List (1.2)\n3. Set (1.2)\n4. SortedSet (1.2)\n5. NavigableSet (1.6)\n6. Queue (1.5)\n7. Map (1.2)\n8. SortedMap (1.2)\n9. NavigableMap (1.6)\n\nThere are **9** key interfaces in the Collection Framework.', 'source_documents': [Document(id='d29d861a-efb7-47fa-8a39-3877511797db', metadata={'source': 'java_collections.pdf'}, page_content='are\n\nrecommended.\n\nrecommended.\n\n4. Arrays can hold only homogeneous\n\n4. Collections can hold both homogeneous and\n\nelements.\n\nheterogeneous elements.\n\n5. Arrays are not implemented based on any DS. So, readymade method support is not available.\n\n5. Every collection class is implemented based on some standard DS. So, readymade method support is available.\n\n6. Arrays can hold primitive types and\n\n6. Collections can hold onl

In [61]:
print(response["result"])

According to the provided text, the 9 Key Interfaces of Collection Framework are:

1. Collection (1.2)
2. List (1.2)
3. Set (1.2)
4. SortedSet (1.2)
5. NavigableSet (1.6)
6. Queue (1.5)
7. Map (1.2)
8. SortedMap (1.2)
9. NavigableMap (1.6)

There are **9** key interfaces in the Collection Framework.
